# Notebook 16: MechInt Pipeline & Database Orchestration
**Building Production-Grade Mechanistic Interpretability Workflows**

## What You'll Learn

In this advanced notebook, you'll master:

- **Database Architecture**: Hybrid HDF5 + SQLite storage for heterogeneous results
- **Content-Based Caching**: SHA256 hashing for automatic deduplication
- **Pipeline Orchestration**: Multi-stage analysis workflows with dependency management
- **Parallel Execution**: Maximize throughput with concurrent analysis
- **Query Optimization**: Fast retrieval across thousands of experiments
- **Comparative Analysis**: Systematic model evaluation and benchmarking
- **Reproducibility**: Provenance tracking and checkpoint/resume

**Prerequisites**: 
- Notebooks 01-15 (understanding of all analysis methods)
- Python 3.8+, PyTorch
- Basic database concepts

**Time Estimate**: 60-90 minutes

---

## Why Pipeline & Database Infrastructure?

### The Challenge

Mechanistic interpretability generates **massive heterogeneous data**:

```
Single Model Analysis:
├── Circuits: Graphs (nodes, edges, scores)
├── Activations: 4D tensors (batch × layers × neurons × time)
├── Energy Cascades: Time series (100K+ points)
├── Hamiltonian Fields: Vector fields (N×D×T)
├── Biophysical: Spike trains, voltage traces
└── Criticality: Power law distributions
```

**Problems**:
- 🔴 Scattered results across filesystems
- 🔴 Duplicate computations (wasted $$)
- 🔴 No systematic comparisons
- 🔴 Manual workflow coordination
- 🔴 Poor reproducibility

### The Solution

**MechIntDatabase** + **MechIntPipeline**:

```
Database Architecture:
├── SQLite: Metadata, tags, provenance
├── HDF5: Large arrays (compression, chunking)
├── SHA256: Content-based deduplication
└── LRU Cache: In-memory result reuse

Pipeline Features:
├── Modular Stages: SAE, Circuits, Dynamics, etc.
├── Dependency DAG: Automatic ordering
├── Parallel Execution: ThreadPoolExecutor
├── Auto-Checkpointing: Resume from failures
└── Report Generation: HTML/PDF/Markdown
```

### Key Papers

- **H5py**: Collette (2013) - "Python and HDF5"
- **Content Addressability**: Git Internals, Merkle Trees
- **Workflow Systems**: Prefect, Luigi, Airflow
- **Provenance**: Davidson & Freire (2008) - "Provenance and Scientific Workflows"

---

In [ ]:
# Imports
import sys
import tempfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Dict, List, Optional, Any
import warnings
import time
warnings.filterwarnings('ignore')

# Bokeh for interactive visualization
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import column, row, gridplot
from bokeh.models import HoverTool, ColorBar, LinearColorMapper, Tabs, Panel
from bokeh.palettes import Viridis256, Spectral11, Category20, RdYlGn11
from bokeh.transform import cumsum
from bokeh.io import export_png
import pandas as pd

# Enable Bokeh in notebook
output_notebook()

# Import database and pipeline
from neuros_mechint.database import MechIntDatabase
from neuros_mechint.pipeline import MechIntPipeline, PipelineConfig

# Import result types
from neuros_mechint.results import (
    MechIntResult,
    CircuitResult,
    DynamicsResult,
    InformationResult,
    AlignmentResult,
    FractalResult,
    ResultCollection
)
from neuros_mechint.results_extended import (
    BiophysicalResult,
    InterventionResult,
    CriticalityResult,
    MultifractalResult
)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("✓ All imports successful!")
print("✓ Bokeh interactive plotting enabled")

---

## Part 1: Database Fundamentals

### 1.1 Hybrid Storage Architecture

The database uses a **hybrid approach** for optimal performance:

```
MechIntDatabase/
├── metadata.db          ← SQLite (fast queries)
├── data/                ← HDF5 files (arrays)
│   ├── circuits/
│   ├── dynamics/
│   └── biophysical/
└── thumbnails/          ← Optional previews
```

**Why hybrid?**
- **SQLite**: Blazing fast tag/metadata queries (milliseconds)
- **HDF5**: Efficient array storage with compression (10-100x smaller)
- **Best of both worlds**: Fast retrieval + compact storage

---

In [ ]:
# Create database with custom configuration
temp_dir = tempfile.mkdtemp()

db = MechIntDatabase(
    root_dir=temp_dir,
    auto_cache=True,           # Enable content-based caching
    max_cache_size_gb=10.0,    # 10 GB max cache
    compression=6,              # HDF5 compression level (0-9)
    verbose=True,
    memory_cache_size=256       # Keep 256 results in memory
)

print(f"Database created at: {temp_dir}")
print(f"\nConfiguration:")
print(f"  - Auto-caching: {db.auto_cache}")
print(f"  - Max cache: {db.max_cache_size_gb} GB")
print(f"  - Compression: Level {db.compression}")
print(f"  - Memory cache: {db._memory_cache_size} results")
print(f"\nStorage layout:")
print(f"  - Metadata DB: {db.metadata_db}")
print(f"  - Data directory: {db.data_dir}")
print(f"  - Thumbnails: {db.thumbnails_dir}")

### 1.2 Content-Based Deduplication

The database uses **SHA256 content hashing** to prevent duplicate computations:

```python
hash = SHA256(method + data + metadata)
```

**Benefits**:
- ✅ Automatic deduplication (save compute $$)
- ✅ Structural sharing (reuse results)
- ✅ Git-like content addressing

---

In [ ]:
# Create sample results to demonstrate storage
print("Creating sample mechanistic interpretability results...\n")

# Circuit result
circuit_result = CircuitResult(
    method="ACDC",
    data={
        'nodes': ['input.0', 'layer1.fc', 'layer1.relu', 'layer2.fc', 'layer2.relu', 'output.0'],
        'edges': [
            ('input.0', 'layer1.fc'),
            ('layer1.fc', 'layer1.relu'),
            ('layer1.relu', 'layer2.fc'),
            ('layer2.fc', 'layer2.relu'),
            ('layer2.relu', 'output.0')
        ],
        'importance_scores': {
            'input.0': 0.95,
            'layer1.fc': 0.88,
            'layer1.relu': 0.76,
            'layer2.fc': 0.82,
            'layer2.relu': 0.69,
            'output.0': 0.91
        }
    },
    metadata={
        'model': 'resnet18_variant',
        'task': 'image_classification',
        'threshold': 0.05,
        'iterations': 15
    },
    metrics={
        'n_nodes': 6,
        'n_edges': 5,
        'avg_importance': 0.835,
        'circuit_complexity': 2.5
    }
)

# Store with tags
circuit_id = db.store(
    result=circuit_result,
    tags=['acdc', 'circuits', 'resnet18', 'image_classification', 'experiment_alpha']
)

print(f"✓ Circuit result stored")
print(f"  ID: {circuit_id}")
print(f"  Tags: {db.get_tags(circuit_id)}")
print(f"  Nodes: {circuit_result.metrics['n_nodes']}")
print(f"  Edges: {circuit_result.metrics['n_edges']}")

In [ ]:
# Demonstrate deduplication
print("\nTesting content-based deduplication...\n")

# Store the same result again (different tags)
duplicate_id = db.store(
    result=circuit_result,  # Identical content
    tags=['acdc', 'duplicate_test', 'resnet18']  # Different tags
)

print(f"Original ID:  {circuit_id}")
print(f"Duplicate ID: {duplicate_id}")
print(f"\nSame ID? {circuit_id == duplicate_id} ✓")
print(f"\nMerged tags: {sorted(db.get_tags(circuit_id))}")
print(f"\n💡 Database automatically detected duplicate content!")
print(f"   Tags were merged, no redundant storage.")

In [ ]:
# Store diverse result types
print("\nStoring diverse mechanistic interpretability results...\n")

# Biophysical result
biophysical_result = BiophysicalResult(
    method="HodgkinHuxley",
    data={'time': np.arange(0, 100, 0.1)},
    metadata={'neuron_type': 'pyramidal', 'compartments': 3},
    metrics={'spike_count': 12, 'mean_isi': 8.3, 'cv_isi': 0.15},
    voltages=np.random.randn(1000, 3) * 10 - 65,
    spike_times=[np.array([10.2, 18.5, 26.8, 35.1]), np.array([12.1, 20.3, 28.6])],
    spike_rate=12.0,
    atp_levels=np.random.rand(1000) * 1.5 + 1.5
)

biophys_id = db.store(
    result=biophysical_result,
    tags=['biophysical', 'hodgkin_huxley', 'pyramidal', 'experiment_alpha']
)

# Criticality result
criticality_result = CriticalityResult(
    method="AvalancheAnalysis",
    data={'avalanche_sizes': np.random.power(1.5, 1000)},
    metadata={'network_size': 1000, 'threshold': 0.1},
    metrics={'branching_param': 0.99, 'power_law_exp': 1.48, 'dcc': 0.85},
    branching_parameter=0.99,
    n_avalanches=342,
    size_distribution=np.random.power(1.5, 1000),
    duration_distribution=np.random.exponential(3.5, 1000),
    is_critical=True
)

crit_id = db.store(
    result=criticality_result,
    tags=['criticality', 'avalanches', 'network_dynamics', 'experiment_alpha']
)

# Multifractal result
multifractal_result = MultifractalResult(
    method="MultifractalAnalysis",
    data={'q_values': np.linspace(-5, 5, 50)},
    metadata={'signal_type': 'spike_train', 'sampling_rate': 1000},
    metrics={'hurst_exponent': 0.72, 'multifractality': 0.35},
    spectrum_dimensions=np.random.rand(50) * 0.5 + 1.5,
    singularity_spectrum=np.random.rand(50) * 2.0,
    hurst_exponent=0.72,
    multifractality_degree=0.35
)

multi_id = db.store(
    result=multifractal_result,
    tags=['multifractal', 'hurst', 'scaling', 'experiment_alpha']
)

print(f"✓ Biophysical result: {biophys_id}")
print(f"✓ Criticality result: {crit_id}")
print(f"✓ Multifractal result: {multi_id}")
print(f"\nTotal results in database: {len(db.list_all())}")

### 1.3 Tag-Based Queries

Efficient querying using **tag-based indexing**:

```python
# AND logic (all tags must match)
results = db.query(tags=['experiment_alpha', 'biophysical'])

# Additional filters
results = db.query(method='ACDC', result_type='CircuitResult')
```

**Performance**: O(log n) with B-tree indices

---

In [ ]:
# Query examples
print("Database Query Examples\n")
print("="*60)

# Query by tags
alpha_results = db.query(tags=['experiment_alpha'])
print(f"\n1. Query: tags=['experiment_alpha']")
print(f"   Found: {len(alpha_results)} results")
for result_id in alpha_results:
    result = db.load(result_id)
    print(f"   - {type(result).__name__}: {result.method}")

# Query biophysical results
biophys_results = db.query(tags=['biophysical'])
print(f"\n2. Query: tags=['biophysical']")
print(f"   Found: {len(biophys_results)} results")

# Query with multiple tags (AND logic)
specific_results = db.query(tags=['experiment_alpha', 'criticality'])
print(f"\n3. Query: tags=['experiment_alpha', 'criticality']")
print(f"   Found: {len(specific_results)} results")

# List all unique tags
all_tags = set()
for result_id in db.list_all():
    all_tags.update(db.get_tags(result_id))

print(f"\n4. All unique tags in database:")
print(f"   {sorted(all_tags)}")

print("\n" + "="*60)

In [ ]:
# Interactive visualization of database contents
print("Creating interactive database visualization...\n")

# Gather statistics
result_types = {}
tag_counts = {}

for result_id in db.list_all():
    result = db.load(result_id)
    result_type = type(result).__name__
    result_types[result_type] = result_types.get(result_type, 0) + 1
    
    for tag in db.get_tags(result_id):
        tag_counts[tag] = tag_counts.get(tag, 0) + 1

# Create visualizations
# Plot 1: Result types distribution
p1 = figure(
    title='Result Types in Database',
    width=600,
    height=400,
    x_range=list(result_types.keys()),
    toolbar_location="above"
)

p1.vbar(x=list(result_types.keys()), top=list(result_types.values()),
       width=0.8, color=Spectral11[:len(result_types)], alpha=0.8)
p1.xaxis.major_label_orientation = 0.785
p1.yaxis.axis_label = "Count"

# Plot 2: Tag usage
sorted_tags = sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)[:10]
tags, counts = zip(*sorted_tags)

p2 = figure(
    title='Top 10 Most Used Tags',
    width=600,
    height=400,
    y_range=list(tags),
    toolbar_location="above"
)

p2.hbar(y=list(tags), right=list(counts), height=0.8,
       color=Category20[len(tags)], alpha=0.8)
p2.xaxis.axis_label = "Usage Count"

# Combine plots
layout = row(p1, p2)
show(layout)

print(f"Database Statistics:")
print(f"  Total results: {len(db.list_all())}")
print(f"  Unique result types: {len(result_types)}")
print(f"  Unique tags: {len(tag_counts)}")

---

## Part 2: Pipeline Orchestration

The **MechIntPipeline** coordinates multi-stage analysis workflows with automatic caching, parallel execution, and dependency management.

### Pipeline Modes

**Three preset configurations**:

1. **Quick** (~5-10 min): Basic circuit discovery + SAE
2. **Standard** (~15-30 min): + Dynamics + Biophysical + Criticality
3. **Comprehensive** (~45+ min): All analyses + Comparisons

### Architecture

```
Pipeline Stage DAG:
                    ┌──────────┐
                    │  Inputs  │
                    └────┬─────┘
                         │
         ┌───────────────┼───────────────┐
         │               │               │
    ┌────▼────┐    ┌────▼────┐    ┌────▼────┐
    │   SAE   │    │ Circuits │    │Dynamics │
    └────┬────┘    └────┬────┘    └────┬────┘
         │               │               │
         └───────────────┼───────────────┘
                         │
                    ┌────▼─────┐
                    │ Biophys  │
                    └────┬─────┘
                         │
                    ┌────▼─────┐
                    │  Report  │
                    └──────────┘
```

---

In [ ]:
# Create simple model for pipeline demonstration
class SimpleClassifier(nn.Module):
    """Simple feedforward classifier for demonstration."""
    def __init__(self, input_dim=64, hidden_dim=128, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.fc3(x)

# Create model
model = SimpleClassifier(input_dim=64, hidden_dim=128, output_dim=10).to(device)
model.eval()

# Create sample inputs
batch_size = 32
sample_inputs = torch.randn(batch_size, 64).to(device)

print(f"Model: SimpleClassifier")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Input shape: {sample_inputs.shape}")
print(f"  Output shape: {model(sample_inputs).shape}")

### 2.1 Quick Pipeline Mode

Fast iteration for exploratory analysis:

---

In [ ]:
# Configure quick pipeline
quick_config = PipelineConfig(
    depth='quick',
    enabled_analyses={'sae', 'circuits'},  # Minimal set
    parallel=True,
    use_cache=True,
    verbose=True
)

print("Quick Pipeline Configuration:")
print(f"  Depth: {quick_config.depth}")
print(f"  Analyses: {sorted(quick_config.enabled_analyses)}")
print(f"  Parallel: {quick_config.parallel}")
print(f"  Cache: {quick_config.use_cache}")

# Note: Pipeline execution is simplified for demonstration
# In practice, you would use:
# pipeline = MechIntPipeline(model=model, db_path=temp_dir, config=quick_config)
# results = pipeline.run(inputs=sample_inputs, experiment_tags=['quick_demo'])

print("\n✓ Quick pipeline configured (ready to run)")

### 2.2 Standard Pipeline Mode

Balanced analysis for typical workflows:

---

In [ ]:
# Configure standard pipeline
standard_config = PipelineConfig(
    depth='standard',
    enabled_analyses={'sae', 'circuits', 'dynamics', 'biophysical', 'criticality'},
    parallel=True,
    use_cache=True,
    verbose=True,
    auto_report=False
)

print("Standard Pipeline Configuration:")
print(f"  Depth: {standard_config.depth}")
print(f"  Analyses: {sorted(standard_config.enabled_analyses)}")
print(f"  Parallel: {standard_config.parallel}")
print(f"  Auto-report: {standard_config.auto_report}")

print("\n✓ Standard pipeline configured")

### 2.3 Comprehensive Pipeline Mode

Full analysis suite for publication-quality results:

---

In [ ]:
# Configure comprehensive pipeline
comprehensive_config = PipelineConfig(
    depth='comprehensive',
    enabled_analyses={
        'sae', 'circuits', 'dynamics', 'fractals',
        'biophysical', 'interventions', 'criticality',
        'multifractal', 'alignment', 'info_flow'
    },
    parallel=True,
    max_workers=4,
    use_cache=True,
    verbose=True,
    auto_report=True,
    report_format='html'
)

print("Comprehensive Pipeline Configuration:")
print(f"  Depth: {comprehensive_config.depth}")
print(f"  Analyses: {len(comprehensive_config.enabled_analyses)}")
print(f"  Parallel workers: {comprehensive_config.max_workers}")
print(f"  Auto-report: {comprehensive_config.auto_report}")
print(f"  Report format: {comprehensive_config.report_format}")
print(f"\nEnabled analyses:")
for i, analysis in enumerate(sorted(comprehensive_config.enabled_analyses), 1):
    print(f"  {i:2d}. {analysis}")

print("\n✓ Comprehensive pipeline configured")

---

## Part 3: Grand Workflow Example - Multi-Model Benchmark

Let's demonstrate the full power of the pipeline + database infrastructure with a **comprehensive multi-model benchmarking workflow**.

### Scenario

Compare 5 neural network architectures across all mechanistic interpretability metrics:
- **Wide**: Large hidden layer (256 neurons)
- **Deep**: Many layers (6 layers)
- **Balanced**: Medium width and depth
- **Bottleneck**: Narrow middle layer
- **Skip**: Residual connections

### Metrics to Compare
1. Circuit complexity (nodes, edges)
2. Energy efficiency (dissipation)
3. Biophysical realism (spike rates, ATP)
4. Criticality (branching parameter)
5. Information flow (mutual information)

---

In [ ]:
# Create 5 model variants
print("Creating model zoo for benchmark...\n")

class WideNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)
    
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class DeepNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(64, 80), nn.ReLU(),
            nn.Linear(80, 80), nn.ReLU(),
            nn.Linear(80, 80), nn.ReLU(),
            nn.Linear(80, 80), nn.ReLU(),
            nn.Linear(80, 80), nn.ReLU(),
            nn.Linear(80, 10)
        )
    
    def forward(self, x):
        return self.layers(x)

class BalancedNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.fc3(x)

class BottleneckNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 32)  # Bottleneck
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 128)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        return self.fc4(x)

class SkipNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        h1 = self.relu1(self.fc1(x))
        h2 = self.relu2(self.fc2(h1))
        h2 = h2 + h1  # Skip connection
        return self.fc3(h2)

# Create model zoo
model_zoo = {
    'wide': WideNetwork().to(device),
    'deep': DeepNetwork().to(device),
    'balanced': BalancedNetwork().to(device),
    'bottleneck': BottleneckNetwork().to(device),
    'skip': SkipNetwork().to(device)
}

# Print statistics
print("Model Zoo:")
print("="*60)
for name, model_instance in model_zoo.items():
    model_instance.eval()
    params = sum(p.numel() for p in model_instance.parameters())
    print(f"{name:12s}: {params:7,} parameters")
print("="*60)

In [ ]:
# Simulate comprehensive analysis results for each model
# In practice, this would run the actual pipeline
print("\nSimulating comprehensive analysis for each model...\n")

benchmark_results = {}

for model_name in model_zoo.keys():
    print(f"Analyzing {model_name}...")
    
    # Simulate circuit analysis
    n_nodes = np.random.randint(5, 12)
    n_edges = np.random.randint(4, n_nodes)
    circuit_complexity = np.random.uniform(1.5, 3.5)
    
    # Simulate energy analysis
    energy_efficiency = np.random.uniform(0.6, 0.95)
    total_dissipation = np.random.uniform(0.1, 0.4)
    
    # Simulate biophysical analysis
    spike_rate = np.random.uniform(8.0, 20.0)
    atp_mean = np.random.uniform(1.5, 2.5)
    
    # Simulate criticality analysis
    branching_param = np.random.uniform(0.85, 1.05)
    is_critical = 0.95 <= branching_param <= 1.05
    
    # Simulate information flow
    mutual_info = np.random.uniform(0.5, 1.5)
    
    benchmark_results[model_name] = {
        'n_nodes': n_nodes,
        'n_edges': n_edges,
        'circuit_complexity': circuit_complexity,
        'energy_efficiency': energy_efficiency,
        'total_dissipation': total_dissipation,
        'spike_rate': spike_rate,
        'atp_mean': atp_mean,
        'branching_param': branching_param,
        'is_critical': is_critical,
        'mutual_info': mutual_info
    }
    
print("\n✓ Benchmark complete!")

### Interactive Benchmark Dashboard

Comprehensive visualization of all metrics across models:

---

In [ ]:
# Create comprehensive benchmark dashboard
print("Creating interactive benchmark dashboard...\n")

models_list = list(benchmark_results.keys())

# Plot 1: Circuit Complexity
p1 = figure(
    title='Circuit Complexity',
    width=500,
    height=350,
    x_range=models_list,
    toolbar_location="above"
)

nodes = [benchmark_results[m]['n_nodes'] for m in models_list]
edges = [benchmark_results[m]['n_edges'] for m in models_list]

x = np.arange(len(models_list))
p1.vbar(x=[i-0.2 for i in x], top=nodes, width=0.35, color='steelblue', legend_label='Nodes', alpha=0.8)
p1.vbar(x=[i+0.2 for i in x], top=edges, width=0.35, color='coral', legend_label='Edges', alpha=0.8)
p1.xaxis.ticker = x
p1.xaxis.major_label_overrides = {i: name for i, name in enumerate(models_list)}
p1.yaxis.axis_label = "Count"
p1.legend.location = "top_right"

# Plot 2: Energy Efficiency
p2 = figure(
    title='Energy Efficiency',
    width=500,
    height=350,
    x_range=models_list,
    toolbar_location="above"
)

efficiency = [benchmark_results[m]['energy_efficiency'] for m in models_list]
colors = [RdYlGn11[int(e*10)] for e in efficiency]

p2.vbar(x=models_list, top=efficiency, width=0.7, color=colors, alpha=0.8)
p2.yaxis.axis_label = "Efficiency"
p2.y_range.start = 0
p2.y_range.end = 1

# Plot 3: Biophysical Metrics
p3 = figure(
    title='Biophysical Metrics',
    width=500,
    height=350,
    x_range=models_list,
    toolbar_location="above"
)

spike_rates = [benchmark_results[m]['spike_rate'] for m in models_list]
atp_levels = [benchmark_results[m]['atp_mean'] for m in models_list]

p3.vbar(x=[i-0.2 for i in x], top=spike_rates, width=0.35, color='purple', legend_label='Spike Rate (Hz)', alpha=0.8)
p3.vbar(x=[i+0.2 for i in x], top=atp_levels, width=0.35, color='green', legend_label='ATP (mM)', alpha=0.8)
p3.xaxis.ticker = x
p3.xaxis.major_label_overrides = {i: name for i, name in enumerate(models_list)}
p3.yaxis.axis_label = "Value"
p3.legend.location = "top_right"

# Plot 4: Criticality
p4 = figure(
    title='Branching Parameter (Criticality)',
    width=500,
    height=350,
    x_range=models_list,
    toolbar_location="above"
)

branch_params = [benchmark_results[m]['branching_param'] for m in models_list]
critical_colors = ['green' if 0.95 <= b <= 1.05 else 'orange' if 0.90 <= b <= 1.10 else 'red' for b in branch_params]

p4.vbar(x=models_list, top=branch_params, width=0.7, color=critical_colors, alpha=0.8)
p4.line([-0.5, len(models_list)-0.5], [1.0, 1.0], color='gray', line_dash='dashed', line_width=2, legend_label='Critical (σ=1)')
p4.yaxis.axis_label = "Branching Parameter σ"
p4.y_range.start = 0.7
p4.y_range.end = 1.2
p4.legend.location = "top_right"

# Arrange in grid
grid = gridplot([[p1, p2], [p3, p4]])
show(grid)

print("\n📊 Interactive Dashboard Created!")
print("\nKey Insights:")

# Find best models
best_efficiency = max(models_list, key=lambda m: benchmark_results[m]['energy_efficiency'])
most_critical = min(models_list, key=lambda m: abs(benchmark_results[m]['branching_param'] - 1.0))
best_atp = max(models_list, key=lambda m: benchmark_results[m]['atp_mean'])

print(f"  🏆 Most energy efficient: {best_efficiency}")
print(f"  🎯 Closest to criticality: {most_critical}")
print(f"  💪 Best metabolic health: {best_atp}")

---

## Part 4: Advanced Features

### 4.1 Provenance Tracking

The database automatically tracks analysis lineage:

```python
# Results store parent relationships
db.add_provenance(child_id, parent_id)

# Query analysis chain
ancestors = db.get_provenance(result_id)
```

**Benefits**:
- Reproducibility
- Audit trail
- Dependency tracking

---

In [ ]:
# Demonstrate provenance tracking
print("Provenance Tracking Example\n")
print("="*60)

# The database automatically tracks which results depend on others
# Example: Biophysical analysis might depend on circuit analysis

print("\nProvenance chain for biophysical analysis:")
print(f"  {biophys_id}")
print(f"    └─ depends on: circuit analysis")
print(f"       └─ {circuit_id}")

print("\n💡 Full provenance tracking enables:")
print("   - Reproducible research")
print("   - Audit trails")
print("   - Dependency resolution")
print("   - Cache invalidation")

### 4.2 Performance Optimization

**Caching Benefits**:

```python
# First run: compute from scratch
result = expensive_analysis(model, inputs)  # 10 minutes

# Second run: retrieve from cache
result = expensive_analysis(model, inputs)  # 0.1 seconds ✨
```

**Savings**:
- **10-1000x** faster on cache hits
- **Zero** duplicate computations
- **Automatic** content-based deduplication

---

In [ ]:
# Demonstrate caching performance
print("Caching Performance Demonstration\n")
print("="*60)

# Simulate timing for cached vs uncached
uncached_time = 10.5  # minutes
cached_time = 0.15    # seconds
speedup = (uncached_time * 60) / cached_time

print(f"\nScenario: Comprehensive pipeline on 5 models")
print(f"\nFirst run (no cache):")
print(f"  - Time per model: {uncached_time:.1f} min")
print(f"  - Total time: {uncached_time * 5:.1f} min")

print(f"\nSecond run (with cache):")
print(f"  - Time per model: {cached_time:.2f} sec")
print(f"  - Total time: {cached_time * 5:.2f} sec")

print(f"\n🚀 Speedup: {speedup:.0f}x faster!")
print(f"\n💰 Cost savings:")
print(f"   - GPU time saved: {uncached_time * 5:.1f} min")
print(f"   - Estimated cloud cost savings: ${(uncached_time * 5 / 60) * 2:.2f}")

---

## Summary & Key Takeaways

### What You Learned

1. **Database Architecture**:
   - Hybrid SQLite + HDF5 for optimal performance
   - Content-based SHA256 hashing prevents duplicates
   - Tag-based queries with O(log n) performance
   - LRU memory cache for instant retrieval

2. **Pipeline Orchestration**:
   - Three modes: Quick, Standard, Comprehensive
   - Automatic dependency resolution
   - Parallel execution with ThreadPoolExecutor
   - Checkpoint/resume for long-running jobs

3. **Workflow Benefits**:
   - **10-1000x** speedup from caching
   - **Zero** duplicate computations
   - **Systematic** model comparisons
   - **Reproducible** research

### Best Practices

1. **Tag Strategically**: Use hierarchical tags (`experiment`, `model_name`, `analysis_type`)
2. **Enable Caching**: Always use `use_cache=True` in production
3. **Parallel Execution**: Set `max_workers` based on CPU cores
4. **Query Before Compute**: Check cache before running expensive analyses
5. **Regular Cleanup**: Archive old experiments to manage storage

### Next Steps

- **Notebook 17**: Advanced biophysical modeling integration
- **Production**: Deploy pipeline for real model evaluation
- **Research**: Use infrastructure for systematic MI research
- **Custom Analyses**: Extend pipeline with domain-specific methods

---

## Exercises

### Exercise 1: Custom Pipeline Configuration

Create a custom pipeline focusing on energy analysis:
- Energy cascades
- Hamiltonian decomposition
- Biophysical modeling
- Landauer thermodynamics

**Hint**: Use `PipelineConfig(depth='custom', enabled_analyses={...})`

In [ ]:
# Exercise 1: Create energy-focused pipeline
# TODO: Configure pipeline with energy-related analyses
# TODO: Run on sample model
# TODO: Visualize energy metrics

# Your code here:
pass

### Exercise 2: Batch Model Evaluation

Create 10 model variants and run comprehensive analysis:
- Store all results with consistent tagging
- Query and compare specific metrics
- Create interactive dashboard
- Identify Pareto-optimal models

**Hint**: Use nested loops and `db.query()` for comparisons

In [ ]:
# Exercise 2: Batch evaluation
# TODO: Create 10 model variants
# TODO: Run pipeline on each
# TODO: Store with systematic tags
# TODO: Query and visualize comparisons

# Your code here:
pass

### Exercise 3: Custom Analysis Stage

Implement a custom analysis that integrates with the pipeline:
- Define custom result class
- Create analysis function
- Store in database
- Query and visualize

**Example**: Gradient flow analysis, attention pattern extraction, etc.

In [ ]:
# Exercise 3: Custom analysis
# TODO: Define custom result dataclass
# TODO: Implement analysis function
# TODO: Store results in database
# TODO: Create visualization

# Your code here:
pass

---

**Congratulations!** You've mastered the MechInt pipeline and database infrastructure. You can now:

✅ Build production-grade mechanistic interpretability workflows  
✅ Leverage content-based caching for 10-1000x speedups  
✅ Conduct systematic multi-model evaluations  
✅ Track provenance and ensure reproducibility  
✅ Create interactive benchmark dashboards  

Continue to **Notebook 17: Advanced Biophysical Modeling** to integrate biological realism into your mechanistic interpretability research.

---